# Model Training

### 1.1 Import Data and Required Packages
#### Importing Pandas, Numpy, Matplotlib, Seaborn and Warings Library.

In [18]:
#basic imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
#modelling
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report, confusion_matrix, make_scorer, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn import svm
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

In [2]:
import warnings
warnings.filterwarnings('ignore')

#### Import the CSV Data as Pandas DataFrame

In [3]:
df = pd.read_csv('../notebooks/data/ai4i2020.csv')

#### show top 5 rows

In [4]:
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


#### Dropping unwanted columns

In [5]:
df = df.drop(columns=['UDI', 'Product ID'])

In [6]:
df.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


#### preparing X and Y variables

In [7]:
df.columns

Index(['Type', 'Air temperature [K]', 'Process temperature [K]',
       'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
       'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'],
      dtype='str')

In [8]:
X = df[['Type', 'Air temperature [K]', 'Process temperature [K]','Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']]

In [9]:
print(f"Categories in Type : {df['Type'].unique()}")

Categories in Type : <ArrowStringArray>
['M', 'L', 'H']
Length: 3, dtype: str


In [10]:
y = df['Machine failure']

In [11]:
y.head()

0    0
1    0
2    0
3    0
4    0
Name: Machine failure, dtype: int64

In [12]:
# Create Column Transformer with 3 types of transformers
num_features = X.select_dtypes(include='number').columns
cat_features = X.select_dtypes(exclude='number').columns

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder()

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoding", oh_transformer, cat_features),
        ("StandardScaler", numeric_transformer, num_features)
    ]
)

In [13]:
X = preprocessor.fit_transform(X)

In [14]:
X.shape

(10000, 8)

In [15]:
# separate dataset into train and test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

((8000, 8), (2000, 8))

#### Training models

In [ ]:
# training different model and storing that trained model inside a dict.

# Model list that I am going to train and test for predict machine failure
models = {
    "Logistic Regression": LogisticRegression(),
    "Decision Tree Classifier": DecisionTreeClassifier(),
    "Random Forest Classifier": RandomForestClassifier(),
    "XGBClassifier": XGBClassifier(),
    "Support Vector": svm.SVC(),
    "GaussianNB": GaussianNB()
}

trained_models = {}

# Trained all models at once
for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model

#### Create an Evaluate Function to give all metrics after model Training

In [27]:
for name, model in trained_models.items():
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    print(f"{name}\n")
    
    print("Evaluation matrix and report for training set: ")
    print(f"Confusion matrics for train set: \n{confusion_matrix(y_train, y_train_pred)}\n")
    print(f"Classification Report for train set: \n{classification_report(y_train, y_train_pred)}")
    print("-"*50)
    
    print("Evalution matrix and report for testing set: ")
    print(f"Confusion matrics for test set: \n{confusion_matrix(y_test, y_test_pred)}\n")
    print(f"Classification reprot for test set: \n{classification_report(y_test, y_test_pred)}")
    
        
    print("="*100)
    print("\n")

Logistic Regression

Evaluation matrix and report for training set: 
Confusion matrics for train set: 
[[7701   21]
 [ 228   50]]

Classification Report for train set: 
              precision    recall  f1-score   support

           0       0.97      1.00      0.98      7722
           1       0.70      0.18      0.29       278

    accuracy                           0.97      8000
   macro avg       0.84      0.59      0.64      8000
weighted avg       0.96      0.97      0.96      8000

--------------------------------------------------
Evalution matrix and report for testing set: 
Confusion matrics for test set: 
[[1931    8]
 [  45   16]]

Classification reprot for test set: 
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      1939
           1       0.67      0.26      0.38        61

    accuracy                           0.97      2000
   macro avg       0.82      0.63      0.68      2000
weighted avg       0.97      0.97    